# Daily Challenge: Build a Tiny Agent with Tools (smolagents) - 🧑‍🎓 STUDENT (fill the TODOs)

**Goal:** a beginner agent that calls simple tools with the **smolagents** framework, using no paid APIs and a tiny local model or a stub.

```
question ─► ToolCallingAgent ─► model proposes a tool call ─► kb_lookup / math ─► observation ─► ... ─► final_answer
```
1. A small knowledge base (`kb_snippets`, 6 snippets with sources).
2. Two tools: `KBLookupTool` (keyword search with `[kb:N]` tags) and `MathTool` (add or multiply).
3. A model:
   - **stub (default):** a deterministic smolagents `Model` that emits real tool calls;
   - `TransformersModel("sshleifer/tiny-gpt2")`, as the exercise asks;
   - optionally, a small instruction-tuned model.
4. A `ToolCallingAgent`.
5. Three test questions, with the tool calls and final answers inspected.

In [ ]:
!pip install -q "smolagents[transformers]" wikipedia

In [ ]:
import os, re, json
os.environ["USE_TF"] = "0"
from importlib.metadata import version
import smolagents
from smolagents import Tool, ToolCallingAgent, TransformersModel
from smolagents.models import Model, ChatMessage, ChatMessageToolCall, ChatMessageToolCallFunction, MessageRole
print("smolagents", version("smolagents"), "| transformers", version("transformers"))

## 1. Knowledge base

In [ ]:
# TODO: define 5-8 short snippets, each with an id, a source and a text
kb_snippets = [
    {"id": 1, "source": "course-notes/agents", "text": "An agentic AI loop repeats observe -> think/plan -> act (call a tool) -> observe the result ..."},
    # ... add more
]
print(len(kb_snippets), "snippets")

## 2. Tools
A smolagents tool subclasses `Tool` and declares a `name`, a `description`, typed `inputs` and an `output_type`. The agent shows this schema to the model.

In [ ]:
class KBLookupTool(Tool):
    name = "kb_lookup"
    description = "Look up the course knowledge base; returns matching snippets with [kb:N] tags, or 'NO_MATCH'."
    inputs = {"query": {"type": "string", "description": "Keywords or a question to search for."}}
    output_type = "string"

    def forward(self, query: str) -> str:
        # TODO: return the snippets whose text contains the query keywords, formatted as "[kb:<id>] <text>"
        raise NotImplementedError


class MathTool(Tool):
    name = "math"
    description = "Add or multiply two numbers. Use op='add' or op='multiply'."
    inputs = {
        "a": {"type": "number", "description": "First number."},
        "b": {"type": "number", "description": "Second number."},
        "op": {"type": "string", "description": "Either 'add' or 'multiply'."},
    }
    output_type = "number"

    def forward(self, a: float, b: float, op: str) -> float:
        # TODO: implement add / multiply (raise ValueError for other ops)
        raise NotImplementedError


kb_tool, math_tool = KBLookupTool(), MathTool()

## 3. The model
- **`StubToolModel` (default):** a tiny smolagents `Model` that behaves like a tool-calling LLM.
  - With no observation yet, it chooses `math` (if the question contains numbers plus add/multiply) or `kb_lookup`.
  - Once the tool has answered, it calls `final_answer` with a 2–4 sentence answer: the KB citations are kept, and if the KB returned `NO_MATCH` it says so and proposes a follow-up question.
- **`TransformersModel("sshleifer/tiny-gpt2")`:** the exercise's default "real" model. It is a 2-layer, randomly initialised test model, so it cannot produce valid tool-call JSON, as we show below.

In [ ]:
USE_TINY_GPT2 = False

class StubToolModel(Model):
    """TODO: deterministic stand-in for a tool-calling LLM.
    generate(messages, ...) must return a ChatMessage whose tool_calls contains ONE ChatMessageToolCall:
      - no tool observation yet  -> call "math" (numbers + add/multiply) or "kb_lookup"
      - after an observation     -> call "final_answer" with a short answer (keep the [kb:N] tags)
    """
    def generate(self, messages, stop_sequences=None, response_format=None, tools_to_call_from=None, **kwargs):
        raise NotImplementedError

# TODO: model = TransformersModel(model_id="sshleifer/tiny-gpt2") if USE_TINY_GPT2 else StubToolModel()
model = None

## 4. The agent

In [ ]:
# TODO: create a ToolCallingAgent with the two tools, the model and max_steps=4
agent = None

## 5. Run the 3 test questions and inspect tool calls + final answers

In [ ]:
def run_and_inspect(question):
    # TODO: run the agent, then loop over agent.memory.steps to print each tool call (name + arguments),
    #       the observation and the final answer
    raise NotImplementedError

answers = {q: run_and_inspect(q) for q in ["Add 12 and 30.", "Multiply 7 by 6.", "What is an agentic AI loop?"]}

## Reflection (TODO)
- Which tool did the agent call for each question, and why?
- Did the KB answer include `[kb:N]` citations?
- What happens with `sshleifer/tiny-gpt2` instead of the stub, and why?